# Sideris Fine-Tuning Pipeline v2 (Kaggle GPU T4 x2 / Colab T4)

Train **Sideris Aether** (`Qwen3.5-4B`), **Sideris Lux** (`Qwen3.5-2B`), **Sideris Orion** (`Phi-4-mini-reasoning`), or **Sideris Titan** (`Qwen3.5-9B`) on your verified UC3M Aerospace Engineering dataset (`train_sft_enriched.jsonl`, `train_dpo_enriched.jsonl`, `train_grpo.jsonl`, `eval_holdout.jsonl`, `rewards.py`).

### Before clicking Run All in Kaggle
1. **Session options -> Accelerator:** `GPU T4 x2` (or `GPU P100`)
2. **Session options -> Internet:** `On`
3. **Input -> + Add Input:** attach your **`Sideris-Data`** dataset (Cell 1 auto-discovers its exact mount path)


In [ ]:
# 1. Environment Guard & Auto-Discovery (MUST run first, before importing torch/transformers/unsloth)
import os, pathlib, subprocess

os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Single T4 prevents DataParallel crashes with Unsloth & 4-bit
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ---- User Configuration ----
MODEL_CHOICE   = "lux"         # "lux" | "aether" | "orion" | "titan"
HF_USERNAME    = "SiderisAI"   # Shared SiderisAI Hugging Face org for auto-upload in Cell 8
RUN_DPO        = True           # Stage 2: DPO alignment (1,310 pairs in v2, ~20-30 min)
RUN_GRPO       = False           # Stage 3: GRPO verifiable RL (capped to GRPO_MAX_STEPS)
SFT_EPOCHS     = 2              # 2 epochs over 5,926 rows in v2 (~5.5-6.5 h on T4, safe inside 12 h)
GRPO_MAX_STEPS = 80             # ~20 min on T4; increase if you want longer RL
MAX_SEQ_LEN    = 2048           # Safe for 15 GB T4 VRAM (zero OOM)

RECIPES = {
    "lux":    {"name": "Sideris-Lux",    "base": "Qwen/Qwen3.5-2B",                "fallback": "Qwen/Qwen3-1.7B", "r": 16, "lr": 2e-4},
    "aether": {"name": "Sideris-Aether", "base": "Qwen/Qwen3.5-4B",                "fallback": "Qwen/Qwen3-4B",   "r": 32, "lr": 2e-4},
    "orion":  {"name": "Sideris-Orion",  "base": "microsoft/Phi-4-mini-reasoning", "fallback": "Qwen/Qwen3-4B",   "r": 32, "lr": 1.5e-4},
    "titan":  {"name": "Sideris-Titan",  "base": "Qwen/Qwen3.5-9B",                "fallback": "Qwen/Qwen3-8B",   "r": 32, "lr": 1.5e-4},
}
RECIPE = RECIPES[MODEL_CHOICE]

# Verify GPU is actually attached
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("No GPU found! Open Kaggle 'Session options' (bottom-right) -> Accelerator = 'GPU T4 x2' and Internet = 'On'.")
print("GPU ready:", gpu.stdout.strip().splitlines()[0])

# Auto-discover Sideris-Data anywhere under /kaggle/input or /content
DATA_DIR = None
for root in ("/kaggle/input", "/content", "."):
    p = pathlib.Path(root)
    if not p.exists():
        continue
    for target in ("train_sft_enriched.jsonl", "train_sft.jsonl"):
        hits = list(p.rglob(target))
        if hits:
            DATA_DIR = str(hits[0].parent)
            break
    if DATA_DIR:
        break

if not DATA_DIR:
    raise FileNotFoundError("Could not find train_sft_enriched.jsonl or train_sft.jsonl under /kaggle/input or /content. Attach Sideris-Data via '+ Add Input'.")

OUT_DIR = "/kaggle/working/sideris_out" if os.path.exists("/kaggle/working") else "/content/sideris_out"
os.makedirs(OUT_DIR, exist_ok=True)

print("Recipe   :", RECIPE)
print("DATA_DIR :", DATA_DIR)
print("OUT_DIR  :", OUT_DIR)
for fname in sorted(os.listdir(DATA_DIR)):
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  - {fname} ({os.path.getsize(fpath)/1024:.1f} KB)")


In [ ]:
# 2. Install Compatible Dependencies (Transformers >= 5.2.0 for Qwen3.5 + Unsloth, WITHOUT touching Kaggle's torch)
import subprocess, sys

def pip_run(*args):
    cmd = [sys.executable, "-m", "pip", *args]
    print(">>", " ".join(cmd))
    subprocess.check_call(cmd)

# NOTE: Do NOT install optimum[onnxruntime] here (it downgrades transformers to 4.57 and breaks Qwen3.5!)
# Also remove broken preinstalled xformers mismatch warning on torch 2.11+cu130 if present.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "xformers"], check=False)

pip_run("install", "-q", "--upgrade",
        "transformers>=5.2.0",
        "trl>=0.15.0",
        "peft>=0.15.0",
        "accelerate>=1.5.0",
        "bitsandbytes>=0.45.0",
        "datasets",
        "sympy",
        "sentencepiece",
        "protobuf",
        "huggingface_hub",
        "hf_transfer")

pip_run("install", "-q", "--upgrade", "--no-deps", "unsloth_zoo", "unsloth")

# Verify versions in a fresh subprocess check + current process
import importlib, torch, transformers, trl, peft, bitsandbytes
print(f"torch={torch.__version__} (cuda={torch.cuda.is_available()})")
print(f"transformers={transformers.__version__} | trl={trl.__version__} | peft={peft.__version__} | bnb={bitsandbytes.__version__}")
if tuple(int(x) for x in transformers.__version__.split(".")[:2]) < (5, 2):
    print("NOTICE: transformers was upgraded in this session; if Cell 3 reports an old cached transformers version, click 'Run -> Restart & clear cell outputs' once and re-run from Cell 1 (pip will be instant).")


In [ ]:
# 3. Load Base Model in 4-bit QLoRA (import unsloth FIRST; automatic HF 4-bit PEFT fallback if needed)
import gc, inspect, os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Always import unsloth BEFORE transformers/trl/peft!
USE_UNSLOTH = False
try:
    import unsloth
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
except Exception as e:
    UNSLOTH_AVAILABLE = False
    print("Unsloth import skipped, will use standard HF 4-bit QLoRA:", e)

import torch

if MODEL_CHOICE == "orion":
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)

target_mods = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] + (
    ["qkv_proj", "gate_up_proj"] if MODEL_CHOICE == "orion" else []
)

model = None
tokenizer = None
LOADED_BASE = None

def unwrap_tokenizer(tok):
    # Qwen3.5 VLM processors wrap the text tokenizer under `.tokenizer`
    return getattr(tok, "tokenizer", tok)

for candidate in (RECIPE["base"], RECIPE["fallback"]):
    print(f"\nLoading {candidate} ...")
    if UNSLOTH_AVAILABLE:
        try:
            fp_params = set(inspect.signature(FastLanguageModel.from_pretrained).parameters)
            fp_kwargs = dict(
                model_name=candidate,
                max_seq_length=MAX_SEQ_LEN,
                dtype=None,
                load_in_4bit=True,
            )
            if "text_only" in fp_params and "qwen3.5" in candidate.lower():
                fp_kwargs["text_only"] = True
            m, t = FastLanguageModel.from_pretrained(**fp_kwargs)
            m = FastLanguageModel.get_peft_model(
                m,
                r=RECIPE["r"],
                lora_alpha=RECIPE["r"],
                target_modules=target_mods,
                lora_dropout=0,
                bias="none",
                use_gradient_checkpointing="unsloth",
                random_state=42,
            )
            model, tokenizer = m, unwrap_tokenizer(t)
            USE_UNSLOTH = True
            LOADED_BASE = candidate
            print(f"SUCCESS: Loaded {candidate} via Unsloth FastLanguageModel (4-bit QLoRA)!")
            break
        except Exception as e_u:
            print(f"  Unsloth failed for {candidate}: {type(e_u).__name__}: {str(e_u)[:300]}")
            gc.collect()
            torch.cuda.empty_cache()

    # Fallback: pure HuggingFace Transformers + 4-bit NF4 BitsAndBytes + PEFT LoRA
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        t = AutoTokenizer.from_pretrained(candidate, trust_remote_code=True)
        m = AutoModelForCausalLM.from_pretrained(
            candidate,
            quantization_config=bnb_cfg,
            device_map={"": 0},
            trust_remote_code=True,
        )
        m.config.use_cache = False
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
        lora_cfg = LoraConfig(
            r=RECIPE["r"],
            lora_alpha=RECIPE["r"],
            target_modules=target_mods,
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
        )
        m = get_peft_model(m, lora_cfg)
        model, tokenizer = m, unwrap_tokenizer(t)
        USE_UNSLOTH = False
        LOADED_BASE = candidate
        print(f"SUCCESS: Loaded {candidate} via HuggingFace 4-bit NF4 + PEFT!")
        break
    except Exception as e_hf:
        print(f"  HF fallback failed for {candidate}: {type(e_hf).__name__}: {str(e_hf)[:300]}")
        gc.collect()
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("Could not load primary or fallback model. If transformers was just upgraded from 4.57 to >=5.2, click Session -> Restart once and re-run.")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Active base model:", LOADED_BASE, "| Unsloth:", USE_UNSLOTH)
model.print_trainable_parameters()


In [ ]:
# 4. Stage 1 — Supervised Fine-Tuning (SFT on train_sft_enriched.jsonl, 3,299 UC3M examples)
import inspect, json, os, torch
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

sft_file = os.path.join(DATA_DIR, "train_sft_enriched.jsonl")
if not os.path.exists(sft_file):
    sft_file = os.path.join(DATA_DIR, "train_sft.jsonl")
print("Using SFT dataset:", sft_file)

def format_chat(messages, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt
        )

# Parse JSONL directly in Python so extra metadata keys on enriched rows never cause PyArrow CastError
with open(sft_file, encoding="utf-8") as f:
    raw_rows = [json.loads(line) for line in f if line.strip()]
sft_ds = Dataset.from_list([
    {"text": format_chat(r["messages"], add_generation_prompt=False)}
    for r in raw_rows if r.get("messages")
])
print(f"Prepared {len(sft_ds)} formatted SFT examples.")

sft_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-sft")
use_bf16 = torch.cuda.is_bf16_supported()

# Version-adaptive SFTConfig (works across TRL 0.12 -> 0.25+)
cfg_params = set(inspect.signature(SFTConfig.__init__).parameters)
cfg_kwargs = {
    "output_dir": sft_out,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "num_train_epochs": SFT_EPOCHS,
    "learning_rate": RECIPE["lr"],
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "fp16": not use_bf16,
    "bf16": use_bf16,
    "optim": "paged_adamw_8bit",
    "logging_steps": 10,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "report_to": "none",
    "seed": 42,
}
if "dataset_text_field" in cfg_params:
    cfg_kwargs["dataset_text_field"] = "text"
if "max_length" in cfg_params:
    cfg_kwargs["max_length"] = MAX_SEQ_LEN
elif "max_seq_length" in cfg_params:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LEN
if "packing" in cfg_params:
    cfg_kwargs["packing"] = False
cfg_kwargs = {k: v for k, v in cfg_kwargs.items() if k in cfg_params}

tr_params = set(inspect.signature(SFTTrainer.__init__).parameters)
tr_kwargs = {
    "model": model,
    "train_dataset": sft_ds,
    "args": SFTConfig(**cfg_kwargs),
}
if "processing_class" in tr_params:
    tr_kwargs["processing_class"] = tokenizer
else:
    tr_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**tr_kwargs)
trainer.train()

model.save_pretrained(sft_out)
tokenizer.save_pretrained(sft_out)
active_adapter = sft_out
print("Stage 1 SFT complete! Saved adapter ->", active_adapter)


In [ ]:
# 5. Stage 2 — DPO Preference Alignment (910 pairs: no algebraic skips, no Phase labels, no Socratic leaks)
import gc, inspect, json, os, torch
from datasets import Dataset

dpo_file = os.path.join(DATA_DIR, "train_dpo_enriched.jsonl")
if not os.path.exists(dpo_file):
    dpo_file = os.path.join(DATA_DIR, "train_dpo.jsonl")

if RUN_DPO and os.path.exists(dpo_file):
    print("Using DPO dataset:", dpo_file)
    gc.collect()
    torch.cuda.empty_cache()

    if USE_UNSLOTH:
        try:
            from unsloth import PatchDPOTrainer
            PatchDPOTrainer()
        except Exception as e:
            print("PatchDPOTrainer skipped:", e)

    from trl import DPOConfig, DPOTrainer

    def msg_text(val):
        if isinstance(val, str):
            return val
        if isinstance(val, list) and val:
            return str(val[-1].get("content", "")) if isinstance(val[-1], dict) else str(val[-1])
        return str(val or "")

    with open(dpo_file, encoding="utf-8") as f:
        raw_dpo = [json.loads(line) for line in f if line.strip()]
    dpo_ds = Dataset.from_list([
        {
            "prompt": format_chat(r["prompt"], add_generation_prompt=True) if isinstance(r["prompt"], list) else str(r["prompt"]),
            "chosen": msg_text(r["chosen"]),
            "rejected": msg_text(r["rejected"]),
        }
        for r in raw_dpo
    ])
    print(f"Prepared {len(dpo_ds)} DPO preference pairs.")

    dpo_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-dpo")
    use_bf16 = torch.cuda.is_bf16_supported()
    dpo_cfg_params = set(inspect.signature(DPOConfig.__init__).parameters)
    dpo_cfg_kwargs = {
        "output_dir": dpo_out,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 1,
        "learning_rate": 5e-6,
        "beta": 0.1,
        "fp16": not use_bf16,
        "bf16": use_bf16,
        "optim": "paged_adamw_8bit",
        "logging_steps": 10,
        "save_strategy": "no",
        "report_to": "none",
        "max_length": MAX_SEQ_LEN,
        "max_prompt_length": 1024,
        "seed": 42,
    }
    dpo_cfg_kwargs = {k: v for k, v in dpo_cfg_kwargs.items() if k in dpo_cfg_params}

    dpo_tr_params = set(inspect.signature(DPOTrainer.__init__).parameters)
    dpo_tr_kwargs = {
        "model": model,
        "ref_model": None,
        "train_dataset": dpo_ds,
        "args": DPOConfig(**dpo_cfg_kwargs),
    }
    if "processing_class" in dpo_tr_params:
        dpo_tr_kwargs["processing_class"] = tokenizer
    else:
        dpo_tr_kwargs["tokenizer"] = tokenizer

    dpo_trainer = DPOTrainer(**dpo_tr_kwargs)
    dpo_trainer.train()

    model.save_pretrained(dpo_out)
    tokenizer.save_pretrained(dpo_out)
    active_adapter = dpo_out
    print("Stage 2 DPO complete! Saved adapter ->", active_adapter)
else:
    print("Skipping Stage 2 DPO.")


In [ ]:
# 6. Stage 3 — GRPO Verifiable RL (KaTeX + No-Skip Rigour + Socratic Guard + SymPy)
import gc, inspect, json, os, sys, torch
from datasets import Dataset

grpo_file = os.path.join(DATA_DIR, "train_grpo.jsonl")
rewards_py = os.path.join(DATA_DIR, "rewards.py")

if RUN_GRPO and os.path.exists(grpo_file) and os.path.exists(rewards_py):
    try:
        gc.collect()
        torch.cuda.empty_cache()
        if DATA_DIR not in sys.path:
            sys.path.insert(0, DATA_DIR)
        from rewards import katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward
        from trl import GRPOConfig, GRPOTrainer

        with open(grpo_file, encoding="utf-8") as f:
            raw_grpo = [json.loads(line) for line in f if line.strip()]
        grpo_ds = Dataset.from_list([
            {
                "prompt": format_chat(r["prompt"], add_generation_prompt=True) if isinstance(r["prompt"], list) else str(r["prompt"]),
                "socratic": bool(r.get("socratic", False)),
                "expected_answer": str(r.get("expected_answer") or ""),
                "forbidden_answers": str(r.get("forbidden_answers") or ""),
            }
            for r in raw_grpo
        ])
        print(f"Loaded {len(grpo_ds)} GRPO prompts (running {GRPO_MAX_STEPS} steps).")

        grpo_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-grpo")
        use_bf16 = torch.cuda.is_bf16_supported()
        g_params = set(inspect.signature(GRPOConfig.__init__).parameters)
        g_kwargs = {
            "output_dir": grpo_out,
            "learning_rate": 5e-6,
            "per_device_train_batch_size": 1,
            "gradient_accumulation_steps": 4,
            "num_generations": 4,
            "max_prompt_length": 1024,
            "max_completion_length": 512,
            "max_steps": GRPO_MAX_STEPS,
            "fp16": not use_bf16,
            "bf16": use_bf16,
            "optim": "paged_adamw_8bit",
            "logging_steps": 5,
            "save_strategy": "no",
            "report_to": "none",
            "seed": 42,
        }
        g_kwargs = {k: v for k, v in g_kwargs.items() if k in g_params}

        gt_params = set(inspect.signature(GRPOTrainer.__init__).parameters)
        gt_kwargs = {
            "model": model,
            "reward_funcs": [katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward],
            "train_dataset": grpo_ds,
            "args": GRPOConfig(**g_kwargs),
        }
        if "processing_class" in gt_params:
            gt_kwargs["processing_class"] = tokenizer
        else:
            gt_kwargs["tokenizer"] = tokenizer

        grpo_trainer = GRPOTrainer(**gt_kwargs)
        grpo_trainer.train()

        model.save_pretrained(grpo_out)
        tokenizer.save_pretrained(grpo_out)
        active_adapter = grpo_out
        print("Stage 3 GRPO complete! Saved adapter ->", active_adapter)
    except Exception as e:
        print(f"GRPO warning (keeping verified SFT+DPO adapter at {active_adapter}):", type(e).__name__, e)
else:
    print("Skipping Stage 3 GRPO.")


In [ ]:
# 7. Qualitative & KaTeX Gate Evaluation on eval_holdout.jsonl
import json, os, sys, torch

if USE_UNSLOTH:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
else:
    model.eval()
    model.config.use_cache = True

eval_file = os.path.join(DATA_DIR, "eval_holdout.jsonl")
if os.path.exists(eval_file):
    if DATA_DIR not in sys.path:
        sys.path.insert(0, DATA_DIR)
    from rewards import katex_syntax_reward
    samples = [json.loads(l) for l in open(eval_file, encoding="utf-8") if l.strip()][:5]
    for s in samples:
        prompt = format_chat(s["messages"][:-1], add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=384, temperature=0.3, do_sample=True, pad_token_id=tokenizer.pad_token_id)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        k_score = katex_syntax_reward([reply])[0]
        print(f"=== {s['id']} ({s['subject']} | {s['task']}) | KaTeX reward: {k_score:+.2f} ===")
        print(reply[:550], "\n")


In [ ]:
# 8. Export Final Weights (LoRA + Merged 16-bit + GGUF Q4_K_M) & Optional Hugging Face Push
import os

final_lora = os.path.join(OUT_DIR, f"{RECIPE['name']}-final-lora")
merged_dir = os.path.join(OUT_DIR, f"{RECIPE['name']}-merged-16bit")
gguf_dir   = os.path.join(OUT_DIR, f"{RECIPE['name']}-gguf")

model.save_pretrained(final_lora)
tokenizer.save_pretrained(final_lora)
print("Saved final LoRA adapter ->", final_lora)

if USE_UNSLOTH:
    try:
        model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
        print("Saved merged 16-bit model ->", merged_dir)
    except Exception as e:
        print("Merged 16-bit export skipped:", e)
    try:
        model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")
        print("Saved GGUF Q4_K_M ->", gguf_dir)
    except Exception as e:
        print("GGUF export skipped (LoRA & merged 16-bit are safe):", e)
else:
    try:
        merged = model.merge_and_unload()
        merged.save_pretrained(merged_dir, safe_serialization=True)
        tokenizer.save_pretrained(merged_dir)
        print("Saved merged 16-bit model ->", merged_dir)
    except Exception as e:
        print("Merge skipped (final LoRA adapter is safe):", e)

# Auto-upload to Hugging Face if Kaggle secret HF_TOKEN is configured and HF_USERNAME is set
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if hf_token and HF_USERNAME and HF_USERNAME != "YOUR_HF_USER":
    from huggingface_hub import HfApi
    api = HfApi(token=hf_token)
    repo_id = f"{HF_USERNAME}/{RECIPE['name']}"
    api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
    upload_src = merged_dir if os.path.isdir(merged_dir) else final_lora
    api.upload_folder(folder_path=upload_src, repo_id=repo_id)
    print(f"Uploaded model -> https://huggingface.co/{repo_id}")
    if os.path.isdir(gguf_dir):
        gguf_repo = f"{HF_USERNAME}/{RECIPE['name']}-GGUF"
        api.create_repo(repo_id=gguf_repo, exist_ok=True, private=True)
        api.upload_folder(folder_path=gguf_dir, repo_id=gguf_repo)
        print(f"Uploaded GGUF -> https://huggingface.co/{gguf_repo}")
else:
    print("Done! All outputs are saved under:", OUT_DIR)
    print("Tip: add Kaggle secret HF_TOKEN + set HF_USERNAME in Cell 1 for automatic Hugging Face upload.")
